# Imports: dash, pandas, plotly.express, os, sys

In [1]:
## imports
from dash import Dash, dcc, html, Input, Output, State, no_update, ctx
import dash_bootstrap_components as dbc
import pandas as pd
import os, sys
sys.path.append(os.path.abspath("../"))
from src.plots import fig_plotter
from src.layout import layout

# Get the data

In [2]:
# get the data
df = pd.read_csv("../data/automobile_sales.csv")
df.head()

,Date,Year,Month,Recession,Consumer_Confidence,Seasonality_Weight,Price,Advertising_Expenditure,Competition,GDP,Growth_Rate,unemployment_rate,Automobile_Sales,Vehicle_Type,City
0,1980-01-31,1980,Jan,1,108.24,0.45,27704,1417.5,7,60.22,0.01,5.4,220.0,SmallFamilyCar,Georgia
1,1980-01-31,1980,Jan,1,108.24,0.45,77270,763.7,7,60.22,0.01,5.4,72.0,Sports,Georgia
2,1980-01-31,1980,Jan,1,108.24,0.36,19665,1417.5,7,60.22,0.01,5.4,238.0,SuperMiniCar,Georgia
3,1980-01-31,1980,Jan,1,108.24,0.38,36986,1417.5,7,60.22,0.01,5.4,224.0,MediumFamilyCar,Georgia
4,1980-02-29,1980,Feb,1,98.75,0.46,26609,2773.4,4,45.99,-0.31,4.8,280.0,SmallFamilyCar,New York


# Data manipulation before visualization

In [3]:
# data manipulation
df["Month"] = pd.Categorical(
    df["Month"],
    categories=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
    ordered=True
)

df["Vehicle_Type"] = (df["Vehicle_Type"]
	.str.split("Car")
	.str[0]
	.str.replace(r'(?<!^)(?=[A-Z])', " ", regex=True))

df.columns = df.columns.str.replace("_", " ", regex=False)
df.head()

,Date,Year,Month,Recession,Consumer Confidence,Seasonality Weight,Price,Advertising Expenditure,Competition,GDP,Growth Rate,unemployment rate,Automobile Sales,Vehicle Type,City
0,1980-01-31,1980,Jan,1,108.24,0.45,27704,1417.5,7,60.22,0.01,5.4,220.0,Small Family,Georgia
1,1980-01-31,1980,Jan,1,108.24,0.45,77270,763.7,7,60.22,0.01,5.4,72.0,Sports,Georgia
2,1980-01-31,1980,Jan,1,108.24,0.36,19665,1417.5,7,60.22,0.01,5.4,238.0,Super Mini,Georgia
3,1980-01-31,1980,Jan,1,108.24,0.38,36986,1417.5,7,60.22,0.01,5.4,224.0,Medium Family,Georgia
4,1980-02-29,1980,Feb,1,98.75,0.46,26609,2773.4,4,45.99,-0.31,4.8,280.0,Small Family,New York


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2112 entries, 0 to 2111
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   Date                     2112 non-null   str     
 1   Year                     2112 non-null   int64   
 2   Month                    2112 non-null   category
 3   Recession                2112 non-null   int64   
 4   Consumer Confidence      2112 non-null   float64 
 5   Seasonality Weight       2112 non-null   float64 
 6   Price                    2112 non-null   int64   
 7   Advertising Expenditure  2112 non-null   float64 
 8   Competition              2112 non-null   int64   
 9   GDP                      2112 non-null   float64 
 10  Growth Rate              2112 non-null   float64 
 11  unemployment rate        2112 non-null   float64 
 12  Automobile Sales         2112 non-null   float64 
 13  Vehicle Type             2112 non-null   object  
 14  City               

# Initiate app via dash

In [5]:
### app initiation
app = Dash(__name__,
		   external_stylesheets=[dbc.themes.BOOTSTRAP],
		   assets_folder=os.path.abspath("../assets"),
		   suppress_callback_exceptions=True, 
		   # use_pages=True,
		   # pages_folder=os.path.abspath("../pages"),
		  )

# Dress the app with the layout: layout codes in "../src/layout"

In [6]:
### master layout: "../src/layout"
app.layout = layout()

# Callback decorator and function for the app: plot codes in "../src/plots"

In [7]:
## callback decorator: 7 input / 8 output
@app.callback(
	Output("year_dropdown", "options"), Output("selected", "children"), Output("plots", "children"), Output("plots", "style"),
	Output("submit", "disabled"), Output("recession_radio", "value"), Output("submit", "n_clicks"), Output("filter", "children"),
	Input("recession_radio", "value"), Input("year_dropdown", "value"), Input("submit", "n_clicks"),
	Input("plot1", "clickData"), Input("plot2", "clickData"), Input("plot3", "clickData"), Input("plot4", "clickData")
)

## callback function
def items_and_plot(recession, year, button_click, click1, click2, click3, click4):
	## initial styles for plot area
	selected= "Select recession type!"
	plots_children = [dcc.Graph(id="plot1"), dcc.Graph(id="plot2"), dcc.Graph(id="plot3"), dcc.Graph(id="plot4")]
	plots_style_grid=dict(display="grid", gridTemplate="repeat(2, 1fr) / repeat(2, 1fr)", height=500)
	plots_style_none=dict(display="none", gridTemplate="repeat(2, 1fr) / repeat(2, 1fr)", height=500)
	reset_disabled_True = True
	reset_disabled_False = False
	recession_radio_value_None = None
	n_clicks0 = 0
	cross_filter_string = "Interact with the charts to cross-filter!"

	## initial controls of radio items, dropdown and reset button and preparation of data IAW the user selection
	# reset button
	if not button_click in [None, 0]:
		return [],  selected, plots_children, plots_style_none, reset_disabled_True, recession_radio_value_None, n_clicks0, ""

	if recession in [None, ""]:
		# selected= "Select recession type!"
		return no_update, selected, no_update, plots_style_none, no_update, no_update, no_update, ""
		
	# radio item initiation and data preparation
	if recession == "Yes":
		rec = df.query("Recession==1")
	elif recession == "No":
		rec = df.query("Recession==0")
	else:	
		rec = df
	years = sorted(rec["Year"].unique())
	options = [dict(label=year, value=year) for year in years]

	# dropdown initiation
	if year in [None, ""]:
		selected= f"Recession: '{recession}' - Select year!"
		return options, selected, no_update, plots_style_none, no_update, no_update, no_update, ""

	# data preparation: continued
	rec_year = rec.query("Year == @year")

	# when the plots are displayed: cross-filter between plots
	# triggered = ctx.triggered_id
	click_data = None
	if click1:
		click_data = click1["points"][0]["x"]
	if click2:
		click_data = click2["points"][0]["x"]
	if click3:
		click_data = click3["points"][0]["x"]
	if click4:
		click_data = click4["points"][0]["label"]

	# data preparation: continued and plotting
	rec_year_click = None
	if click_data:
		column_name = rec_year.columns[rec_year.eq(click_data).any(axis=0)][0]
		rec_year_click = rec_year[rec_year[column_name] == click_data]

	if rec_year_click is not None and not rec_year_click.empty:
		plot1, plot2, plot3, plot4 = fig_plotter(rec_year_click) # "../src/plots"
		selected = f"Recession: '{recession}' - Year: {year} - {column_name}: '{click_data}'"
	else:
		plot1, plot2, plot3, plot4 = fig_plotter(rec_year) # "../src/plots"
		selected = f"Recession: '{recession}' - Year: {year}"

	## plots
	plots = [dcc.Graph(id="plot1", figure=plot1), dcc.Graph(id="plot2", figure=plot2), dcc.Graph(id="plot3", figure=plot3), dcc.Graph(id="plot4", figure=plot4)]

	## final return
	return options, selected, plots, plots_style_grid, reset_disabled_False, no_update, no_update, cross_filter_string

# Run the app

In [8]:
## run the app
if __name__ == "__main__":
	app.run(debug=True, jupyter_mode="external", port=8050)

Dash app running on http://127.0.0.1:8050/
